# clear_badminton - training

Trains `yolov8n-p2` (stride-4 head) on the `clear-badminton-court-v2` Roboflow
export: 84 hand-boxed frames from `dataset_court1` and `dataset_court2`.

- **Gate**: measured median shuttle 13.1 px (court1) / 13.2 px (court2), both in
  the 8-16 px band, so `p2` rather than stock yolov8n. Same call as the previous
  run, re-measured on this dataset rather than assumed.
- **Never downscale.** At imgsz 640 a 13 px shuttle becomes ~7 px, which is below
  YOLOv8's finest stride of 8. `imgsz=1280` is the native long side.
- **Splits come from the export, not from Ultralytics.** They were assigned by
  150-frame blocks at extraction time; a random re-split here would put
  near-duplicate neighbours in train and valid and the val mAP would stop
  meaning anything.
- **Mono footage**: hue/saturation augmentation disabled - those channels carry
  no information in this venue's lighting.

### Known weakness of this dataset

84 boxes across 84 images, and **zero background images**. The failure this
model is meant to fix was 10,463 oversized false positives on this same footage;
negatives are the direct remedy and there are none here. Expect the val numbers
to look better than real-world behaviour. Judge the result by the false-positive
count in `detect_video.py`, not by mAP.

Runtime -> Change runtime type -> **T4 GPU** before running.

In [ ]:
!nvidia-smi
!pip -q install ultralytics==8.4.14

## Upload the dataset

Run this cell, then choose `clear-badminton-court-v2.yolov8.zip` (the Roboflow
export, straight from `setup/datasets/`).

In [ ]:
from google.colab import files
uploaded = files.upload()
print(list(uploaded))

In [ ]:
import glob, os, zipfile, yaml
from collections import Counter

ZIP = "clear-badminton-court-v2.yolov8.zip"
ROOT = "/content/clear_badminton_yolo"
with zipfile.ZipFile(ZIP) as zf:
    zf.extractall(ROOT)

# Roboflow writes relative paths ("../train/images") that only resolve from the
# yaml's own directory. Absolute paths survive whatever cwd Ultralytics picks.
DATA = f"{ROOT}/data.yaml"
with open(DATA, "w") as fh:
    yaml.safe_dump({"path": ROOT, "train": "train/images", "val": "valid/images",
                    "test": "test/images", "nc": 1, "names": ["shuttlecock"]},
                   fh, sort_keys=False)
print(open(DATA).read())

## Refuse to spend GPU time on a leaked split

Frames 6 apart share a court, lighting and players; only the shuttle has moved.
If a block straddles train and valid, val measures memorisation. The extraction
assigned whole 150-frame blocks to one split each, so the check is that no
`(video, block)` pair appears in two splits.

In [ ]:
BLOCK = 150

def blocks_of(split):
    out = set()
    for p in glob.glob(f"{ROOT}/{split}/images/*.jpg"):
        stem = os.path.basename(p)
        video = "dataset_court1" if "court1" in stem else "dataset_court2"
        # Roboflow appends a hash suffix: dataset_court1_000750_jpg.rf.<hash>.jpg
        frame = int(stem.split("_")[2])
        out.add((video, frame // BLOCK))
    return out

tr, va, te = blocks_of("train"), blocks_of("valid"), blocks_of("test")
assert not (tr & va), f"blocks shared by train and valid: {sorted(tr & va)}"
assert not (tr & te), f"blocks shared by train and test: {sorted(tr & te)}"
assert not (va & te), f"blocks shared by valid and test: {sorted(va & te)}"

counts = {s: len(glob.glob(f"{ROOT}/{s}/images/*.jpg")) for s in ("train", "valid", "test")}
assert counts == {"train": 59, "valid": 18, "test": 7}, counts

import cv2
sizes = Counter(cv2.imread(p).shape[:2] for p in glob.glob(f"{ROOT}/*/images/*.jpg"))
assert list(sizes) == [(720, 1280)], f"unexpected resolution: {sizes}"

boxes = sum(len([l for l in open(p).read().splitlines() if l.strip()])
            for p in glob.glob(f"{ROOT}/*/labels/*.txt"))
print("counts:", counts)
print("blocks:", {"train": len(tr), "valid": len(va), "test": len(te)})
print("boxes:", boxes, " resolution:", dict(sizes))
print("OK: no block spans two splits")

## Train

In [ ]:
from ultralytics import YOLO

IMGSZ = 1280  # native long side - do NOT lower this
PROJECT = "/content/runs/clear_badminton"  # absolute: a relative project nests under Ultralytics' runs_dir

model = YOLO("yolov8-p2.yaml")

results = model.train(
    data=DATA,
    epochs=120,          # 59 training images; the previous run's 80 stopped improving early
    imgsz=IMGSZ,
    batch=8,
    workers=2,
    seed=0,
    deterministic=True,
    project=PROJECT,
    name="p2-native",
    patience=30,         # wider than the 20 before: tiny datasets plateau noisily
    cache=False,
    # --- mono-specific augmentation ---
    hsv_h=0.0,     # no hue information exists in this footage
    hsv_s=0.0,     # no saturation information exists either
    scale=0.25,    # default 0.5 would annihilate a 13 px object
    flipud=0.0,    # gravity is real; shuttles fall
    fliplr=0.5,    # the court is roughly symmetric
    mosaic=1.0,
    close_mosaic=10,
    plots=True,
    val=True,
)

## Evaluate on the held-out test split

`best.pt` is selected on `valid`, so its valid score is optimistic by
construction. `test` was never seen by the checkpoint selection, which makes it
the only honest number in the run - on 7 images, so read it as a smoke test, not
a measurement.

In [ ]:
best = YOLO(f"{results.save_dir}/weights/best.pt")
m = best.val(data=DATA, split="test", imgsz=IMGSZ, plots=False)
print(f"test  P={m.box.mp:.3f}  R={m.box.mr:.3f}  mAP50={m.box.map50:.3f}  mAP50-95={m.box.map:.3f}")

## Download

Unzip into `setup/runs/clear_badminton/` locally, then point `detect_video.py`
at the weights:

```
python scripts/feeder_court/detect_video.py --show --full     --weights runs/clear_badminton/p2-native/weights/best.pt     --videos datasets/vid_source/clear_badminton_dataset_for_collab/dataset_court1.mp4
```

The number that matters is the `oversized` column. The current model scores
10,463 on this clip.

In [ ]:
import shutil
RUN = str(results.save_dir)
shutil.make_archive("/content/clear_badminton_train", "zip", RUN)
print(sorted(os.listdir(RUN)))
files.download("/content/clear_badminton_train.zip")